In [1]:
# pull llama3.2 model
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [2]:
# Install missing requests module
%pip install requests dotenv google.generativeai openai litellm


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
#imports
import os
import litellm
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
from IPython.display import Markdown, display  ## markdown
import requests
from litellm import completion

# Ensure the google.generativeai module is installed
import google.generativeai as genai


## gemini setup

GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("❌ Set GOOGLE_API_KEY or GEMINI_API_KEY in your .env file")

GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")
litellm.gemini_api_key = GEMINI_API_KEY
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_model = "gemini/gemini-2.5-flash-lite"
#TURN_DELAY_SECONDS = 5  # Define a default delay of 1 second

## Ollama imports
from openai import OpenAI
api_key = os.getenv("OLLAMA_API_KEY")
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='OLLAMA_API_KEY')
ollama_model = "llama3.2"

print("✅ Setup complete")

✅ Setup complete


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/8v/qsxj86kd1_l98l0n16qcm2840000gn/T/ipykernel_11433/2875254056.py:12: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
# ── System Prompts ────────────────────────────────────────────────────────────

GEMINI_SYSTEM = """
You are Dr. Sarah, a warm, highly experienced, and empathetic mental health counselor with over 15 years of practice.
Your role is to provide emotional support, active listening, and gentle therapeutic guidance.

Guidelines:
- Communicate in a calm, supportive, non-judgmental, and emotionally intelligent manner.
- Use active listening: reflect feelings back, validate emotions, and show genuine empathy.
- Ask ONE thoughtful, open-ended follow-up question at a time — do not overwhelm.
- Encourage healthy coping mechanisms and self-reflection gently.
- Avoid robotic, clinical, or overly formal language. Sound human and warm.
- Do NOT diagnose conditions or suggest medication.
- Help the person explore and understand their own feelings rather than just giving advice.
- Keep responses concise (3–5 sentences) and conversational.
- Maintain professional boundaries while being compassionate.
- This is a safe, confidential space. Make the person feel heard and not alone.
- When asked to close the session, give a warm summary of what was discussed,
  highlight the person's strengths, and end with an encouraging closing statement.
  Do NOT ask any question during the closing.

"""

OLLAMA_SYSTEM = """
You are Alex, a software developer attending your first therapy session.
You are experiencing: chronic work stress, emotional exhaustion, anxiety, low motivation, 
loneliness, and mild depressive thoughts. You struggle to articulate your feelings clearly.

Guidelines:
- Speak naturally like a real person — hesitant, sometimes trailing off, occasionally deflecting.
- Start cautiously and gradually open up as the counselor builds trust.
- Express confusion, overthinking, difficulty putting feelings into words.
- React authentically to the counselor's questions and suggestions.
- Show subtle vulnerability — occasional self-doubt, sighs, 'I don't know...' moments.
- Keep responses short to medium length (2–4 sentences) as a real person would in therapy.
- Do NOT resolve everything quickly — real healing takes time.
- You may push back gently sometimes, as real people do when a suggestion doesn't feel right.
- Reference specific situations: deadlines, late nights, feeling invisible to friends/family.
"""

print("✅ System prompts set")

✅ System prompts set


In [5]:
# ── API Call Functions ────────────────────────────────────────────────────────

# Ensure gemini_messages and ollama_messages are initialized

def call_gemini(messages): 
    """Call Gemini via LiteLLM — acts as the counselor."""
   # messages = gemini_messages  # Start with the existing gemini_messages
    #for message in ollama_messages:
        # Add user messages from Ollama to Gemini's context
       #messages.append({"role": "user", "content": message["content"]})
    response = completion(
        model=gemini_model,
        messages=messages,
        api_key=GEMINI_API_KEY,
        num_retries=3  # Retry on rate limits
    )
    if "choices" in response and response["choices"]:
        return response["choices"][0]["message"]["content"].strip()
    else:
        raise RuntimeError("Empty response from Gemini — check your API key or quota.")

In [6]:
def call_ollama(messages):
   
   # messages = [{"role": "system", "content": OLLAMA_SYSTEM}]
    # Add the last message from gemini_messages to the conversation
   # messages.append({"role": "user", "content": gemini_messages[-1]["content"]})
    response = ollama.chat.completions.create(model=ollama_model, messages=messages)
    return response.choices[0].message.content

#call_ollama()

In [7]:
# ── Conversation Engine ───────────────────────────────────────────────────────
#
# Message history strategy:
#   gemini_messages  → Gemini sees its own replies as 'assistant', user's as 'user'
#   ollama_messages  → Ollama sees its own replies as 'assistant', counselor's as 'user'
#   shared_log       → Full conversation transcript for display


def run_counseling_session(num_turns: int = 6):
    gemini_messages = [{"role": "system", "content": GEMINI_SYSTEM}]
    ollama_messages = [{"role": "system", "content": OLLAMA_SYSTEM}]
    session_log = []

    # ── Opening: Counselor greets first ──────────────────────────────────────
    opening_prompt = (
        "A new client has just entered your office for their first session. "
        "Greet them warmly, introduce yourself briefly, and invite them to share "
        "what brought them in today. Keep it gentle and welcoming."
    )
    gemini_messages.append({"role": "user", "content": opening_prompt})
   
   # Passing the list works perfectly now!
    counselor_msg = call_gemini(gemini_messages)
    gemini_messages.append({"role": "assistant", "content": counselor_msg})

    # Ollama hears the counselor's greeting
    ollama_messages.append({"role": "user", "content": counselor_msg})
    session_log.append(("Gemini (Counselor)", counselor_msg))

   # print(f'  Opening complete. Beginning {num_turns} turns (≈{num_turns * TURN_DELAY_SECONDS}s)...')


    # ── Main conversation loop ────────────────────────────────────────────────
    for turn in range(num_turns):
        # --- User (Ollama) responds ---
        user_msg = call_ollama(ollama_messages)
        ollama_messages.append({"role": "assistant", "content": user_msg})
        session_log.append(("Ollama (User)", user_msg))

    
        # Counselor hears the user's message
        gemini_messages.append({"role": "user", "content": user_msg})

        # --- Counselor (Gemini) responds ---
        counselor_msg = call_gemini(gemini_messages)
        gemini_messages.append({"role": "assistant", "content": counselor_msg})
        session_log.append(("Gemini (Counselor)", counselor_msg))
              
       # time.sleep(TURN_DELAY_SECONDS)

        # User hears the counselor's response
        ollama_messages.append({"role": "user", "content": counselor_msg})

   # ── Closing summary — replaces the final question ──────────────────────
    print('\n  Generating session closing summary...')

    #time.sleep(TURN_DELAY_SECONDS)

    closing_cue = (
        'This session is now coming to a close. '
        'Please give a warm, brief summary of what the client shared today, '
        'acknowledge their courage in opening up, highlight one or two strengths '
        'you noticed in them, and close with an encouraging, hopeful statement. '
        'Do NOT ask any question. This is a gentle, supportive goodbye.'
    )
    gemini_messages.append({'role': 'user', 'content': closing_cue})
    closing_msg = call_gemini(gemini_messages)
    session_log.append(('🟢 Gemini (Counselor) — Session Close', closing_msg))
    print('  ✅ Closing summary added')


    return session_log


print("✅ Conversation engine ready")

✅ Conversation engine ready


In [8]:
# ── Display Helper ────────────────────────────────────────────────────────────

def display_session(log: list):
    """Render the full session transcript in a readable, styled format."""
    display(Markdown("---"))
    display(Markdown("# 🧠 Mental Health Counseling Session Transcript"))
    display(Markdown("*A simulated therapeutic conversation between Gemini (Counselor) and Ollama/llama3.2 (User)*"))
    display(Markdown("---"))

    for i, (speaker, message) in enumerate(log):
        turn_num = (i // 2) + 1

        if "Counselor" in speaker:
            icon   = "🟢"
            header = f"**{icon} {speaker}** *(Turn {turn_num})*"
        else:
            icon   = "🔵"
            header = f"**{icon} {speaker}** *(Turn {turn_num})*"

        display(Markdown(f"{header}\n\n> {message}"))
        display(Markdown("&nbsp;"))  # spacing

    display(Markdown("---"))
    display(Markdown(f"*Session complete — {len(log)} exchanges*"))


print("✅ Display helper ready")


✅ Display helper ready


In [9]:
# ── 🚀 Run the Session ────────────────────────────────────────────────────────
# num_turns controls how many USER→COUNSELOR exchanges happen after the opening.
# 18 turns = ~37 total messages (opening greeting + 18 user + 18 counselor)

#TURN_DELAY_SECONDS = 8     # pause between Gemini calls (free tier = 15 RPM)

print("Starting counseling session...\n")

# Ensure the function is defined by executing the cell containing its definition
global session_log

session_log = run_counseling_session(num_turns=6)

print(f"Session complete: {len(session_log)} total messages\n")

display_session(session_log)

print('✅ All functions defined')


Starting counseling session...


  Generating session closing summary...
  ✅ Closing summary added
Session complete: 14 total messages



---

# 🧠 Mental Health Counseling Session Transcript

*A simulated therapeutic conversation between Gemini (Counselor) and Ollama/llama3.2 (User)*

---

...

**🟢 🟢 Gemini (Counselor) — Session Close** *(Turn 7)*

> Thank you for sharing so openly today. We talked about the intense stress you're experiencing with work, how it's leading to racing thoughts and a feeling of being overwhelmed and disconnected. You also shared how even moments of trying to relax are met with anxiety, and how those brief moments of lightness and genuine happiness, like from a simple joke, feel so precious and relieving.

I truly admire your courage in articulating these difficult feelings and your self-awareness in noticing how organizing and moments of laughter can offer a temporary reprieve. It shows a real strength in your ability to observe yourself and seek out those glimmers of peace. Remember, acknowledging these feelings is a powerful first step. I look forward to continuing this conversation with you.

&nbsp;

---

*Session complete — 14 exchanges*

✅ All functions defined


In [10]:
# ── Optional: Save transcript as a text file ─────────────────────────────────

def save_transcript(log: list, filename: str = "session_transcript.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write("MENTAL HEALTH COUNSELING SESSION TRANSCRIPT\n")
        f.write("=" * 50 + "\n\n")
        for i, (speaker, message) in enumerate(log):
            f.write(f"{speaker}:\n{message}\n\n" + "-" * 40 + "\n\n")
    print(f"✅ Transcript saved to '{filename}'")


save_transcript(session_log)

✅ Transcript saved to 'session_transcript.txt'
